In [1]:
#import
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
#データの読み込み
df=pd.read_csv('sukkiri-ml-codes (1)\datafiles\Boston.csv')

In [3]:
#データの前処理
df=df.fillna(df.mean(numeric_only=True))  #欠損値補完
df=df.drop([76],axis=0)  #外れ値の削除

y=df[['PRICE']]
x=df.loc[:,['RM','PTRATIO','LSTAT']]  #特徴量抽出

#標準化
sc=StandardScaler()
sc_x=sc.fit_transform(x)
sc2=StandardScaler()
sc_y=sc2.fit_transform(y)

In [4]:
from sklearn.preprocessing import PolynomialFeatures

pf=PolynomialFeatures(degree=2,include_bias=False)
pf_x=pf.fit_transform(sc_x)  #二乗列と交互作用特徴量の追加
pf_x.shape  #行数と列数の確認

(99, 9)

In [5]:
pf.get_feature_names_out()

array(['x0', 'x1', 'x2', 'x0^2', 'x0 x1', 'x0 x2', 'x1^2', 'x1 x2',
       'x2^2'], dtype=object)

In [6]:
#線形回帰で過学習が起こるのかの確認
from sklearn.linear_model import LinearRegression

x_train,x_test,y_train,y_test=train_test_split(pf_x,sc_y,test_size=0.3,random_state=0)
model=LinearRegression()
model.fit(x_train,y_train)

print(model.score(x_train,y_train))
print(model.score(x_test,y_test))

0.8710525685992707
0.7854929935582586


In [8]:
#リッジ回帰で過学習が起こるのかの確認
from sklearn.linear_model import Ridge

#モデルの作成
ridgeModel=Ridge(alpha=10)
ridgeModel.fit(x_train,y_train)  #学習
print(ridgeModel.score(x_train,y_train))
print(ridgeModel.score(x_test,y_test))

0.8607320524729508
0.8458730019328173


In [9]:
#alpha（Fの係数）を0.1~20まで変化させて実験
maxScore=0
maxIndex=0

for i in range(1,2001):
    num=i/100
    ridgeModel=Ridge(random_state=0,alpha=num)
    ridgeModel.fit(x_train,y_train)  #学習

    result=ridgeModel.score(x_test,y_test)
    if result>maxScore:
        maxScore=result
        maxIndex=num

print(f'Index={maxIndex},Score={maxScore}')

Index=17.62,Score=0.8528754801497631


In [12]:
print(sum(abs(model.coef_)[0]))  #線形回帰の係数（絶対値）の合計
print(sum(abs(ridgeModel.coef_)))  #リッジ回帰の合計

1.556674598328836
1.2152800824026813


In [13]:
#ラッソ回帰で過学習が起きているか確認
from sklearn.linear_model import Lasso

x_train,x_test,y_train,y_test=train_test_split(pf_x,sc_y,test_size=0.3,random_state=0)
model=Lasso(alpha=0.1)
model.fit(x_train,y_train)

print(model.score(x_train,y_train))
print(model.score(x_test,y_test))

0.8224680202036665
0.858846785318774


In [16]:
weight=model.coef_  #係数抜き出す
pd.Series(weight,index=pf.get_feature_names_out())

x0       0.409426
x1      -0.083104
x2      -0.287714
x0^2     0.150001
x0 x1   -0.000000
x0 x2   -0.037450
x1^2    -0.000000
x1 x2    0.000000
x2^2     0.000000
dtype: float64

In [17]:
#回帰木を用いた学習
x2=df.loc[:,'ZN':'LSTAT']
y2=df['PRICE']
x2_train,x2_test,y2_train,y2_test=train_test_split(x2,y2,test_size=0.3,random_state=0)

from sklearn.tree import DecisionTreeRegressor

model=DecisionTreeRegressor(max_depth=10,random_state=0)
model.fit(x2_train,y2_train)
model.score(x2_test,y2_test)

0.6628396651964714

In [18]:
pd.Series(model.feature_importances_,index=x2.columns)

ZN         0.002948
INDUS      0.013474
CHAS       0.000000
NOX        0.019912
RM         0.773511
AGE        0.008514
DIS        0.012882
RAD        0.000563
TAX        0.000321
PTRATIO    0.010119
B          0.012568
LSTAT      0.145188
dtype: float64